# B2-024-gpu-scientific-ml-capstone — Practice p26 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** scientific-ml-inverse-problems

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import time

MAX_STEPS = 2000
MAX_SECONDS = 15.0

## Solution

**Idea: amortized initialization + physics refinement.** The forward model is known exactly: sensor $i$ reads $m_i=q/(\lVert s-p_i\rVert^2+h^2)\,(1+0.03\varepsilon_i)$. Taking logs, $\log m_i=\log q-\log(\lVert s-p_i\rVert^2+h^2)+\eta_i$ with $\eta_i\approx0.03\varepsilon_i$, roughly Gaussian with constant scale, so the maximum-likelihood source minimizes the squared **log residual**
$$\sum_{i=1}^8\Big(\log\hat q-\log(\lVert\hat s-p_i\rVert^2+h^2)-\log m_i\Big)^2 .$$
This is a 3-parameter nonconvex fit per reading vector; it needs a good starting point. An MLP on standardized log magnitudes (trained on the training rows) supplies that start in one forward pass, and a short batched Adam refinement of $(\hat x,\hat y,\log\hat q)$ on the log residual finishes the job. One refinement optimizer handles a whole batch of readings, so it costs one budget step per update.

**Budget plan** (one `StepBudget(2000)` created before any fitting): MLP training 1,000 steps; refinement 100 steps on validation; the fixed-start alternative 100 steps on validation; refinement 100 steps inside the test call. Total 1,300 steps.

In [ ]:
budget = capstone_data.StepBudget(MAX_STEPS)
train_x, train_y = capstone_data.train_rows("inverse")
val_x, val_y = capstone_data.val_rows("inverse")
SENSORS = capstone_data.SENSORS_XY            # float64 (8, 2)
H2 = capstone_data.SENSOR_HEIGHT ** 2
SEAM = []


def physics_refine(magnitudes, init, steps, lr, budget):
    # Batched maximum-likelihood fit of (x, y, log q) to the log readings; one optimizer step per update.
    s = init[:, :2].detach().double().clone().requires_grad_(True)
    log_q = torch.log(init[:, 2].detach().double().clamp_min(0.1)).requires_grad_(True)
    optimizer = budget.wrap(torch.optim.Adam([s, log_q], lr=lr))
    log_m = torch.log(magnitudes.double())
    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        r2 = ((s[:, None, :] - SENSORS[None, :, :]) ** 2).sum(dim=-1)
        residual = log_q[:, None] - torch.log(r2 + H2) - log_m
        (residual ** 2).sum().backward()
        optimizer.step()
    return torch.cat([s.detach(), torch.exp(log_q.detach())[:, None]], dim=1).float()


class AmortizedPhysicsInverse:
    def __init__(self, budget, hidden=64, train_steps=1000, refine_steps=100, refine_lr=3e-3, seed=SEED):
        self.budget, self.hidden, self.train_steps = budget, hidden, train_steps
        self.refine_steps, self.refine_lr, self.seed = refine_steps, refine_lr, seed

    def fit(self, train_x, train_y):
        SEAM.extend(capstone_data.split_of("inverse", row) for row in train_x)
        log_m = torch.log(train_x)
        self.mu, self.sd = log_m.mean(dim=0), log_m.std(dim=0)          # training statistics only
        torch.manual_seed(self.seed)
        h = self.hidden
        self.net = nn.Sequential(nn.Linear(8, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 3))
        optimizer = self.budget.wrap(torch.optim.Adam(self.net.parameters(), lr=3e-3))
        z = (log_m - self.mu) / self.sd
        for _ in range(self.train_steps):
            optimizer.zero_grad(set_to_none=True)
            F.mse_loss(self.net(z), train_y).backward()
            optimizer.step()
        self.net.eval()
        return self

    def initial(self, x):
        with torch.no_grad():
            return self.net((torch.log(x) - self.mu) / self.sd)

    def predict(self, x):
        return physics_refine(x, self.initial(x), self.refine_steps, self.refine_lr, self.budget)


approach = AmortizedPhysicsInverse(budget).fit(train_x, train_y)
steps_after_fit = budget.used

val_err = lambda pred: capstone_data.position_error_per_example(pred, val_y).mean().item()
val_pred = approach.predict(val_x)
VAL = {"amortized MLP + physics refinement (chosen)": val_err(val_pred)}
VAL["MLP alone (same network, no refinement)"] = val_err(approach.initial(val_x))
VAL["grid Tikhonov (committed baseline method)"] = val_err(capstone_data.tikhonov_grid_predict(val_x))
fixed_start = torch.tensor([[0.5, 0.5, 1.25]]).repeat(val_x.shape[0], 1)             # domain center, mid strength
VAL["physics refinement from a fixed start (100 steps)"] = val_err(physics_refine(val_x, fixed_start, 100, 1e-2, budget))

def true_residual_floor(x, y):
    r2 = ((y[:, None, :2].double() - SENSORS[None]) ** 2).sum(-1)
    return ((torch.log(y[:, 2:].double()) - torch.log(r2 + H2) - torch.log(x.double())) ** 2).mean().sqrt().item()

for name, e in VAL.items():
    print(f"validation mean position error, {name}: {e:.4f}")
print(f"RMS log residual of the true validation sources (noise floor): {true_residual_floor(val_x, val_y):.4f}")
print(f"steps used so far: {budget.used} (fit {steps_after_fit}); elapsed {budget.elapsed():.2f}s")

**Locked test score (one call, after all fitting and selection).** The refinement inside `predict` is wrapped by the same budget.

In [ ]:
baseline = capstone_data.baseline_score("inverse")
assert capstone_data.test_call_count("inverse") == 0
test = capstone_data.final_test_score(approach.predict, n_boot=1000, alpha=0.05, task="inverse")
assert capstone_data.test_call_count("inverse") == 1
elapsed = budget.elapsed()
print(f"test mean position error {test.score:.4f}, 95% CI [{test.ci_low:.4f}, {test.ci_high:.4f}]; baseline {baseline.score:.4f} [{baseline.ci_low:.4f}, {baseline.ci_high:.4f}]")
print(f"budget used {budget.used} of {MAX_STEPS}; elapsed {elapsed:.2f}s")

## Writeup

**Approach.** A two-stage estimator that uses the known physics. Stage 1 is an `8 → 64 → 64 → 3` MLP on standardized log magnitudes (statistics from the training rows), trained for 1,000 full-batch Adam steps on the 800 training pairs; logs are used because the noise is multiplicative and the strength is then an additive offset. Stage 2 takes the MLP's $(\hat x,\hat y,\hat q)$ as the starting point and runs 100 batched Adam steps on $(\hat x,\hat y,\log\hat q)$ minimizing the squared log residual between the forward model's readings and the observed ones, which is the maximum-likelihood fit under the 3% multiplicative noise. The MLP gets the fit into the right basin instantly; the refinement removes the regression error the MLP cannot (its finite capacity and training set), so the final estimate is limited by the sensor noise rather than by the network.

**Alternatives considered** (validation numbers printed above). (1) *The grid-Tikhonov baseline method*: about $0.21$ mean position error; 8 readings cannot pin down 81 grid densities and the top-3 centroid is biased toward grid nodes, so it was never competitive. (2) *The MLP alone*: about $0.012$, roughly 60% worse than with refinement, because the remaining error is learnable-function approximation error that the per-example physics fit removes. (3) *Physics refinement from a fixed start* (domain center, mid strength) with the same 100-step budget: clearly worse than starting from the MLP, because from a poor start 100 steps are not enough to converge for sources far from the center (in an exploratory validation run, about 300 steps from the fixed start came close to the same optimum, so the MLP's value is the start it provides: it saves most of the refinement budget).

**Evaluation.** Fitting used only `train_rows("inverse")` (the seam hashed all 800 rows as `"train"`); every comparison above used the 200 validation rows; the locked test rows were reached once, through `final_test_score`, after all choices. The approach's test mean position error and its 95% bootstrap interval are printed above; the committed grid-Tikhonov baseline scores $0.2137$ under the same protocol, so the improvement is more than an order of magnitude and the two intervals are far apart. Total cost: 1,300 of 2,000 budgeted optimizer steps and a few seconds of wall-clock time.

**Limitations.** (i) The refinement assumes the forward model is exact (sensor positions, height, single point source); a second source or a miscalibrated sensor would bias it, which I would test by simulating such cases (`simulate` with a perturbed forward model) and checking the residual and the error. (ii) The fit is nonconvex: a start in the wrong basin (e.g. near the domain edge or a mirror-symmetric configuration) can converge to a wrong local minimum; I would test this by plotting error against true position on validation and checking cases whose final log residual is far above the noise floor. (iii) Uncertainty is reported only through the test bootstrap over examples, not per prediction.

### Answer check

In [ ]:
assert test.score < baseline.score
assert budget.used <= MAX_STEPS
assert elapsed < MAX_SECONDS
assert len(SEAM) == 800 and not (set(SEAM) & {"val", "test"})
assert capstone_data.test_call_count("inverse") == 1
assert val_pred.shape == (200, 3)
assert VAL["amortized MLP + physics refinement (chosen)"] == min(VAL.values())     # chosen on validation